In [1]:
import sys
sys.path.append("..")

import src.globals as g
from pathlib import Path
g.HF_CACHE_DIR = Path("/work3/s215225/hf_cache")
import numpy as np

import matplotlib.pyplot as plt
from src.hidden_objects_dataset import (
    HiddenObjectsImageClassLevelFast,
    BoxGaussianHeatmap,
    BoxBetaModeHeatmap,
)

In [ ]:
ds = HiddenObjectsImageClassLevelFast(split="test", image_size=512)

rng = np.random.default_rng(seed=42)
indices = rng.choice(len(ds), size=20, replace=False).tolist()
samples = [ds[i] for i in indices]
print(f"Loaded {len(samples)} random samples")

In [ ]:
from matplotlib.patches import Rectangle

gaussian_fn  = BoxGaussianHeatmap()
beta_mode_fn = BoxBetaModeHeatmap()

def plot_boxes_labeled(sample, ax):
    image = sample["image"].permute(1, 2, 0).numpy()
    image = (image - image.min()) / (image.max() - image.min())
    ax.imshow(image)
    ax.axis("off")

    boxes   = sample["boxes"].numpy()
    confs   = sample["confidences"].numpy()
    rewards = sample["image_reward_scores"].numpy()
    labels  = sample["labels"].numpy()

    order = np.argsort(rewards)
    keep  = np.concatenate([order[:10], order[-10:]])

    for idx in keep:
        x, y, w, h = boxes[idx]
        color = "lime" if labels[idx] == 1 else "red"
        ax.add_patch(Rectangle((x, y), w, h, linewidth=1.5, edgecolor=color, facecolor="none"))
        ax.text(x, y - 3, f"c={confs[idx]:.2f} r={rewards[idx]:.2f}", fontsize=9, color=color,
                verticalalignment="bottom", clip_on=True)

fig, axes = plt.subplots(20, 4, figsize=(20, 100))

for i, (sample, idx) in enumerate(zip(samples, indices)):
    mean_r = sample["image_reward_scores"].mean().item()
    image  = sample["image"].permute(1, 2, 0).numpy()
    image  = (image - image.min()) / (image.max() - image.min())

    axes[i, 0].imshow(image)
    axes[i, 0].set_title(f"{sample['class']} | mean_r={mean_r:.2f}", fontsize=10)
    axes[i, 0].axis("off")

    plot_boxes_labeled(sample, ax=axes[i, 1])
    axes[i, 1].set_title("10 highest + 10 lowest r per image (green=pos, red=neg)", fontsize=10)

    axes[i, 2].imshow(gaussian_fn(sample).numpy(), cmap="jet", vmin=0, vmax=1)
    axes[i, 2].set_title("BoxGaussian", fontsize=10)
    axes[i, 2].axis("off")

    axes[i, 3].imshow(beta_mode_fn(sample).numpy(), cmap="jet", vmin=0, vmax=1)
    axes[i, 3].set_title("BoxBetaMode", fontsize=10)
    axes[i, 3].axis("off")

plt.tight_layout()
plt.show()